In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 #######################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습 곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 모델 성능 평가 ########################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 #########################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습 모델 ########################
# 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier 
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

# 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원 축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관 규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주형 데이터, 순위X)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 오버 샘플링
from imblearn.over_sampling import SMOTE

# 객체를 파일에 저장
import pickle

# 문장 데이터 관련
# 단어 토큰화
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
# 정규식 사용
import re
# 불용어 처리
from nltk.corpus import stopwords
# 어간 추출
from nltk.stem import PorterStemmer
# 표제어 추출
from nltk.stem import WordNetLemmatizer

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

### 어간 추출과 표제어 추출 예시

In [3]:
words = ['running', 'flies', 'happily', 'ate', 'better']

In [4]:
# 어간 추출
stemmer = PorterStemmer()
# 표제어 추출
lemmatizer = WordNetLemmatizer()

In [10]:
# 추출 한다.
test_dict = {
    '단어' : [],
    '어간추출' : [],
    '표제어추출' : []
}

# 단어의 수 만큼 반복한다.
for word in words :
    # 어간 추출
    stem = stemmer.stem(word)
    # 표제어 추출. 품사 정보를 주면 더 정확해진다. 동시를 뜻하는 v를 설정한다.
    lemma = lemmatizer.lemmatize(word, pos='v') if word =='ate' else lemmatizer.lemmatize(word)

    test_dict['단어'].append(word)
    test_dict['어간추출'].append(stem)
    test_dict['표제어추출'].append(lemma)

test_df = pd.DataFrame(test_dict)
test_df

,단어,어간추출,표제어추출
0,running,run,running
1,flies,fli,fly
2,happily,happili,happily
3,ate,ate,eat
4,better,better,better


### 스팸 문자는 어간 추출을 추천한다.
- 스팸 데이터는 수천 개의 짧은 메시지로 구성된다. 어간 추출은 복잡한 사전 조회사 품사 분석 없이 정해진 규익에 따라 단어 끝을 잘라내기 때문에 처리 속도가 앞도적으로 빠르다.
- 패턴 매칭의 일관성 스팸 분류의 핵심은 단어의 정확한 문법이 아니라 비슷한 단어들이 얼마나 자주 등장하는가 입니다.

In [13]:
from ast import literal_eval

df = pd.read_csv('data/spam5.csv', encoding='latin-1')
df['clean_tokens'] = df['clean_tokens'].apply(literal_eval)
df

,label,text,tokenized_text,clean_text,clean_tokens
0,ham,"Go until jurong point, crazy.. Available only ...","['Go', 'until', 'jurong', 'point', ',', 'crazy...",Go until jurong point crazy Available only in ...,"[Go, jurong, point, crazy, Available, bugis, n..."
1,ham,Ok lar... Joking wif u oni...,"['Ok', 'lar', '...', 'Joking', 'wif', 'u', 'on...",Ok lar Joking wif u oni,"[Ok, lar, Joking, wif, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"['Free', 'entry', 'in', '2', 'a', 'wkly', 'com...",Free entry in a wkly comp to win FA Cup final ...,"[entry, wkly, comp, win, FA, Cup, final, tkts,..."
3,ham,U dun say so early hor... U c already then say...,"['U', 'dun', 'say', 'so', 'early', 'hor', '......",U dun say so early hor U c already then say,"[dun, say, early, hor, c, already, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...","['Nah', 'I', 'do', ""n't"", 'think', 'he', 'goes...",Nah I don t think he goes to usf he lives arou...,"[Nah, think, goes, usf, lives, around, though]"
...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"['This', 'is', 'the', '2nd', 'time', 'we', 'ha...",This is the nd time we have tried contact u U ...,"[time, tried, contact, Pound, prize, claim, ea..."
5568,ham,Will Ì_ b going to esplanade fr home?,"['Will', 'Ì_', 'b', 'going', 'to', 'esplanade'...",Will b going to esplanade fr home,"[b, going, esplanade, fr, home]"
5569,ham,"Pity, * was in mood for that. So...any other s...","['Pity', ',', '*', 'was', 'in', 'mood', 'for',...",Pity was in mood for that So any other suggest...,"[Pity, mood, suggestions]"
5570,ham,The guy did some bitching but I acted like i'd...,"['The', 'guy', 'did', 'some', 'bitching', 'but...",The guy did some bitching but I acted like i d...,"[guy, bitching, acted, like, interested, buyin..."


In [14]:
# 각 단어들의 어간 추출을 위한 함수
def apply_stemming(token_list) :
    # 각 토큰에 대한 어간 추출을 진행하고 다시 리스트로 변환
    return [stemmer.stem(word) for word in token_list]

In [15]:
# 어간추출을 진행한다.
df['stemmed_tokens'] = df['clean_tokens'].apply(apply_stemming)
df

,label,text,tokenized_text,clean_text,clean_tokens,stemmed_tokens
0,ham,"Go until jurong point, crazy.. Available only ...","['Go', 'until', 'jurong', 'point', ',', 'crazy...",Go until jurong point crazy Available only in ...,"[Go, jurong, point, crazy, Available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,Ok lar... Joking wif u oni...,"['Ok', 'lar', '...', 'Joking', 'wif', 'u', 'on...",Ok lar Joking wif u oni,"[Ok, lar, Joking, wif, oni]","[ok, lar, joke, wif, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"['Free', 'entry', 'in', '2', 'a', 'wkly', 'com...",Free entry in a wkly comp to win FA Cup final ...,"[entry, wkly, comp, win, FA, Cup, final, tkts,...","[entri, wkli, comp, win, fa, cup, final, tkt, ..."
3,ham,U dun say so early hor... U c already then say...,"['U', 'dun', 'say', 'so', 'early', 'hor', '......",U dun say so early hor U c already then say,"[dun, say, early, hor, c, already, say]","[dun, say, earli, hor, c, alreadi, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...","['Nah', 'I', 'do', ""n't"", 'think', 'he', 'goes...",Nah I don t think he goes to usf he lives arou...,"[Nah, think, goes, usf, lives, around, though]","[nah, think, goe, usf, live, around, though]"
...,...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"['This', 'is', 'the', '2nd', 'time', 'we', 'ha...",This is the nd time we have tried contact u U ...,"[time, tried, contact, Pound, prize, claim, ea...","[time, tri, contact, pound, prize, claim, easi..."
5568,ham,Will Ì_ b going to esplanade fr home?,"['Will', 'Ì_', 'b', 'going', 'to', 'esplanade'...",Will b going to esplanade fr home,"[b, going, esplanade, fr, home]","[b, go, esplanad, fr, home]"
5569,ham,"Pity, * was in mood for that. So...any other s...","['Pity', ',', '*', 'was', 'in', 'mood', 'for',...",Pity was in mood for that So any other suggest...,"[Pity, mood, suggestions]","[piti, mood, suggest]"
5570,ham,The guy did some bitching but I acted like i'd...,"['The', 'guy', 'did', 'some', 'bitching', 'but...",The guy did some bitching but I acted like i d...,"[guy, bitching, acted, like, interested, buyin...","[guy, bitch, act, like, interest, buy, someth,..."


### 뿌리 찾기
- 어간 추출의 의미..
- 문제 메시지 본문에서 불필요한 단어를 다 솎어낸 뒤, 남은 핵심 단어들의 꼬리(접미사)를 잘라내어 통일하는 과정
- 예) winner, winning, win -> win
- 속도 향상

In [16]:
stemmer = PorterStemmer()
df['final_tokens'] = df['clean_tokens'].apply(lambda x : [stemmer.stem(word) for word in x])
df

,label,text,tokenized_text,clean_text,clean_tokens,stemmed_tokens,final_tokens
0,ham,"Go until jurong point, crazy.. Available only ...","['Go', 'until', 'jurong', 'point', ',', 'crazy...",Go until jurong point crazy Available only in ...,"[Go, jurong, point, crazy, Available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,Ok lar... Joking wif u oni...,"['Ok', 'lar', '...', 'Joking', 'wif', 'u', 'on...",Ok lar Joking wif u oni,"[Ok, lar, Joking, wif, oni]","[ok, lar, joke, wif, oni]","[ok, lar, joke, wif, oni]"
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,"['Free', 'entry', 'in', '2', 'a', 'wkly', 'com...",Free entry in a wkly comp to win FA Cup final ...,"[entry, wkly, comp, win, FA, Cup, final, tkts,...","[entri, wkli, comp, win, fa, cup, final, tkt, ...","[entri, wkli, comp, win, fa, cup, final, tkt, ..."
3,ham,U dun say so early hor... U c already then say...,"['U', 'dun', 'say', 'so', 'early', 'hor', '......",U dun say so early hor U c already then say,"[dun, say, early, hor, c, already, say]","[dun, say, earli, hor, c, alreadi, say]","[dun, say, earli, hor, c, alreadi, say]"
4,ham,"Nah I don't think he goes to usf, he lives aro...","['Nah', 'I', 'do', ""n't"", 'think', 'he', 'goes...",Nah I don t think he goes to usf he lives arou...,"[Nah, think, goes, usf, lives, around, though]","[nah, think, goe, usf, live, around, though]","[nah, think, goe, usf, live, around, though]"
...,...,...,...,...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,"['This', 'is', 'the', '2nd', 'time', 'we', 'ha...",This is the nd time we have tried contact u U ...,"[time, tried, contact, Pound, prize, claim, ea...","[time, tri, contact, pound, prize, claim, easi...","[time, tri, contact, pound, prize, claim, easi..."
5568,ham,Will Ì_ b going to esplanade fr home?,"['Will', 'Ì_', 'b', 'going', 'to', 'esplanade'...",Will b going to esplanade fr home,"[b, going, esplanade, fr, home]","[b, go, esplanad, fr, home]","[b, go, esplanad, fr, home]"
5569,ham,"Pity, * was in mood for that. So...any other s...","['Pity', ',', '*', 'was', 'in', 'mood', 'for',...",Pity was in mood for that So any other suggest...,"[Pity, mood, suggestions]","[piti, mood, suggest]","[piti, mood, suggest]"
5570,ham,The guy did some bitching but I acted like i'd...,"['The', 'guy', 'did', 'some', 'bitching', 'but...",The guy did some bitching but I acted like i d...,"[guy, bitching, acted, like, interested, buyin...","[guy, bitch, act, like, interest, buy, someth,...","[guy, bitch, act, like, interest, buy, someth,..."


In [17]:
df.to_csv('data/spam6.csv', encoding='latin-1', index=False)